# LeetCode 483 — Smallest Good Base bằng Python

- Link: https://leetcode.com/problems/smallest-good-base/
- Độ khó: **Hard**
- Chủ đề: Toán (hệ cơ số, cấp số nhân), Binary Search, Căn bậc `m` nguyên

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestGoodBase(self, n: str) -> str:

```

## 1. Mục tiêu học

1. Hiểu **biểu diễn số trong cơ số `k`** và cách đổi cơ số bằng `%` và `//`.
2. Nhận ra "toàn chữ số 1 trong cơ số `k`" chính là **tổng cấp số nhân** `1 + k + k² + … + k^(m-1)`.
3. Biết **đổi vai trò biến**: thay vì duyệt cơ số `k` (rất nhiều), ta duyệt **số chữ số `m`** (chỉ khoảng 60 giá trị).
4. Dùng **binary search trên đáp án** và **căn bậc `m` nguyên chính xác** (tránh sai số `float`).
5. Chứng minh vì sao với mỗi `m` chỉ có **một ứng viên duy nhất** `k = ⌊n^(1/(m-1))⌋`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho số nguyên `n` dưới dạng chuỗi. Gọi `k >= 2` là một **cơ số tốt (good base)** của `n` nếu khi viết `n` trong cơ số `k`, **mọi chữ số đều là `1`**. Hãy trả về cơ số tốt **nhỏ nhất**, dưới dạng chuỗi.

**Ví dụ 1**

```text
Input:  n = "13"
Output: "3"
Giải thích: 13 trong cơ số 3 là 111  (1·9 + 1·3 + 1 = 13)
```

**Ví dụ 2**

```text
Input:  n = "4681"
Output: "8"
Giải thích: 4681 trong cơ số 8 là 11111  (4096 + 512 + 64 + 8 + 1)
```

**Ví dụ 3**

```text
Input:  n = "1000000000000000000"
Output: "999999999999999999"
Giải thích: trong cơ số n-1, n được viết là 11
```

**Ràng buộc**

- `n` là số nguyên trong khoảng `[3, 10^18]`.
- `n` không có số 0 ở đầu.

Nhận xét: luôn tồn tại đáp án vì `k = n - 1` cho `n = 1·(n-1) + 1` → "11". Câu hỏi là có cơ số **nhỏ hơn** không.

In [1]:
def to_base(n, k):
    """Trả về danh sách chữ số của n trong cơ số k (chữ số cao nhất đứng đầu)."""
    digits = []
    while n > 0:
        digits.append(n % k)
        n //= k
    return digits[::-1]

print("13 base 3   :", to_base(13, 3))
print("4681 base 8 :", to_base(4681, 8))
n = 10**18
print("10^18 base n-1:", to_base(n, n - 1))

13 base 3   : [1, 1, 1]
4681 base 8 : [1, 1, 1, 1, 1]
10^18 base n-1: [1, 1]


## 3. Kiến thức nền

### 3.1. Đổi cơ số bằng `%` và `//`

Muốn biết chữ số cuối của `n` trong cơ số `k`: `n % k`. Bỏ chữ số cuối: `n // k`. Lặp tới khi `n = 0`.

Ví dụ `13` trong cơ số `3`:
- `13 % 3 = 1`, `13 // 3 = 4`
- `4 % 3 = 1`, `4 // 3 = 1`
- `1 % 3 = 1`, `1 // 3 = 0` → dừng. Chữ số (từ cuối lên): `1,1,1` → `111`.

In [2]:
n, k = 13, 3
while n > 0:
    print(f"n={n:>3}: chữ số cuối = n % k = {n % k}, n // k = {n // k}")
    n //= k

n= 13: chữ số cuối = n % k = 1, n // k = 4
n=  4: chữ số cuối = n % k = 1, n // k = 1
n=  1: chữ số cuối = n % k = 1, n // k = 0


### 3.2. Kiểm tra "toàn chữ số 1"

Thay vì lưu cả danh sách, ta kiểm tra ngay: mỗi bước nếu `n % k != 1` thì trả `False`.

In [3]:
def all_ones(n, k):
    while n > 0:
        if n % k != 1:
            return False
        n //= k
    return True

print(all_ones(13, 3), all_ones(13, 2), all_ones(4681, 8), all_ones(7, 2))

True False True True


### 3.3. Tổng cấp số nhân

"`n` có `m` chữ số 1 trong cơ số `k`" nghĩa là:

`n = 1 + k + k² + … + k^(m-1) = (k^m - 1) / (k - 1)`

Ví dụ `k = 3, m = 3`: `1 + 3 + 9 = 13 = (27 - 1)/2`.

Trong Python số nguyên không tràn, nên `k**m` tính chính xác (dù rất lớn).

In [4]:
def geo_sum(k, m):
    total, term = 0, 1
    for _ in range(m):
        total += term
        term *= k
    return total

print(geo_sum(3, 3), (3**3 - 1) // (3 - 1))
print(geo_sum(8, 5), (8**5 - 1) // (8 - 1))

13 13
4681 4681


### 3.4. Số chữ số tối đa: `n.bit_length()`

Cơ số nhỏ nhất là `2`, khi đó số chữ số nhiều nhất. Số chữ số của `n` trong cơ số 2 là `n.bit_length()`. Với `n <= 10^18`, giá trị này `<= 60`. Vậy `m` (số chữ số) chỉ nằm trong `[2, 60]`.

In [5]:
print((10**18).bit_length(), (13).bit_length(), bin(13))

60 4 0b1101


### 3.5. Vì sao không tin `float`? Căn bậc `m` nguyên

`n ** (1/3)` trả về `float` có ~15–16 chữ số có nghĩa. Với `n ~ 10^18` có thể sai lệch 1 đơn vị. Ta tự viết **căn bậc `e` nguyên** bằng binary search: tìm `r` lớn nhất sao cho `r**e <= n`.

In [6]:
def iroot(n, e):
    """Số nguyên r lớn nhất với r**e <= n (n >= 1, e >= 1)."""
    lo, hi = 1, 1 << (n.bit_length() // e + 1)   # hi chắc chắn có hi**e > n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid ** e <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

x = 999999999999999999
print("float:", int((x * x) ** 0.5), " iroot:", iroot(x * x, 2), " đúng:", x)
print(iroot(4681, 4), iroot(13, 2))

float: 1000000000000000000  iroot: 999999999999999999  đúng: 999999999999999999
8 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `n = 13`:

| k | 13 trong cơ số k | toàn 1? |
|---|---|---|
| 2 | 1101 | không |
| 3 | 111 | **có** → đáp án 3 |
| 12 | 11 | có (nhưng lớn hơn) |

Với `n = 21`: cơ số 2 → `10101` (không), 3 → `210`, 4 → `111` (**có**: 16+4+1). Đáp án 4.

Quan sát quan trọng:
- **Cơ số càng nhỏ thì càng nhiều chữ số.** Vậy cơ số tốt nhỏ nhất ứng với **số chữ số lớn nhất** có thể.
- Số chữ số `m` rất ít lựa chọn (≤ 60), còn cơ số `k` có tới `10^18` lựa chọn. Đổi biến duyệt từ `k` sang `m` là chìa khóa.

In [7]:
for n in [13, 21, 31, 4681]:
    for k in range(2, n):
        if all_ones(n, k):
            print(f"n={n}: cơ số tốt nhỏ nhất = {k}, biểu diễn = {to_base(n, k)}")
            break

n=13: cơ số tốt nhỏ nhất = 3, biểu diễn = [1, 1, 1]
n=21: cơ số tốt nhỏ nhất = 4, biểu diễn = [1, 1, 1]
n=31: cơ số tốt nhỏ nhất = 2, biểu diễn = [1, 1, 1, 1, 1]
n=4681: cơ số tốt nhỏ nhất = 8, biểu diễn = [1, 1, 1, 1, 1]


### 4.1. Bảng "những số có toàn chữ số 1"

Hãy lật ngược câu hỏi: thay vì hỏi "`n` có cơ số tốt nào?", ta **sinh** các số `f(k, m) = 1 + k + … + k^(m-1)` cho `k` và `m` nhỏ. Mỗi ô trong bảng là một số `n` mà cơ số `k` tốt với `m` chữ số.

Đọc bảng theo hàng (cố định `m`): giá trị tăng rất nhanh theo `k`. Đọc theo cột (cố định `k`): tăng theo `m`. Điều này gợi ý **binary search** theo `k` khi cố định `m`.

In [8]:
print("m\\k " + "".join(f"{k:>8}" for k in range(2, 9)))
for m in range(2, 7):
    print(f"{m:>4} " + "".join(f"{geo_sum(k, m):>8}" for k in range(2, 9)))

m\k        2       3       4       5       6       7       8
   2        3       4       5       6       7       8       9
   3        7      13      21      31      43      57      73
   4       15      40      85     156     259     400     585
   5       31     121     341     781    1555    2801    4681
   6       63     364    1365    3906    9331   19608   37449


Nhìn bảng: `13` xuất hiện ở hàng `m = 3`, cột `k = 3`; `31` xuất hiện **hai lần** (`m=5, k=2` và `m=3, k=5`) — khi đó đáp án là `k` nhỏ hơn, ứng với `m` lớn hơn. Đây chính là Insight 4 ở mục 6.

Thống kê nhỏ: trong các số từ 3 đến 10 000, bao nhiêu số chỉ có cơ số tốt là `n-1`?

In [9]:
from collections import Counter
special = Counter()
for k in range(2, 101):
    for m in range(3, 15):
        v = geo_sum(k, m)
        if v <= 10000:
            special[v] += 1
print("Số lượng n <= 10000 có cơ số tốt khác n-1:", len(special), "trên", 10000 - 2)
print("Vài số có >= 2 cơ số tốt (ngoài n-1):", [v for v, c in special.items() if c >= 2])

Số lượng n <= 10000 có cơ số tốt khác n-1: 140 trên 9998
Vài số có >= 2 cơ số tốt (ngoài n-1): [31, 8191]


Kết luận thực tế: **hầu hết** `n` chỉ có đáp án `n - 1`. Brute force vì thế gần như luôn phải chạy hết `n` bước — một lý do nữa để bỏ nó.

## 5. Cách 1 — Brute Force: thử mọi cơ số

**Ý tưởng:** duyệt `k = 2, 3, …, n-1`, kiểm tra `all_ones(n, k)`; gặp đầu tiên thì trả về.

In [10]:
def good_base_brute(n_str):
    n = int(n_str)
    for k in range(2, n):
        if all_ones(n, k):
            return str(k)
    return str(n - 1)

assert good_base_brute("13") == "3"
assert good_base_brute("4681") == "8"
assert good_base_brute("3") == "2"
print([good_base_brute(str(n)) for n in range(3, 20)])

['2', '3', '4', '5', '2', '7', '8', '9', '10', '11', '3', '13', '2', '15', '16', '17', '18']


**Độ phức tạp:** mỗi lần kiểm tra `O(log_k n)`, số `k` tới `n` → tệ nhất `O(n log n)`. Với `n = 10^18` (đáp án thường là `n-1`) phải thử `10^18` cơ số — không thể.

**Nút thắt:** duyệt theo `k` quá nhiều giá trị. Phần lớn `n` không có cơ số tốt nào ngoài `n-1`, nên brute force gần như luôn chạy hết.

In [11]:
import time
t = time.perf_counter()
good_base_brute(str(10**6 + 3))
print(f"n ≈ 10^6: {time.perf_counter() - t:.3f}s  (10^18 sẽ mất hàng triệu năm)")

n ≈ 10^6: 0.053s  (10^18 sẽ mất hàng triệu năm)


## 6. Bước đột phá

**Insight 1 — Duyệt theo số chữ số `m`.** Nếu `k` là cơ số tốt với `m` chữ số thì `n = 1 + k + … + k^(m-1)`. Vì `k >= 2` nên `n >= 2^m - 1`, tức `m <= log2(n+1)`, tức `m <= 60`.

**Insight 2 — Với mỗi `m`, hàm `f(k) = 1 + k + … + k^(m-1)` tăng nghiêm ngặt theo `k`.** Do đó có **nhiều nhất một** `k` thỏa `f(k) = n`, và ta tìm được bằng **binary search**.

**Insight 3 — Ứng viên duy nhất là `k = ⌊n^(1/(m-1))⌋`.** Vì:
- `k^(m-1) < f(k) = n` → `k < n^(1/(m-1))`.
- `n = f(k) < (k+1)^(m-1)` vì khai triển nhị thức `(k+1)^(m-1) = Σ C(m-1, i) k^i` có mọi hệ số `C(m-1,i) >= 1` và có hệ số `> 1` khi `m >= 3`. → `k + 1 > n^(1/(m-1))`.
- Vậy `k < n^(1/(m-1)) < k+1` → `k = ⌊n^(1/(m-1))⌋`. (Khi `m = 2` thì `k = n - 1` trực tiếp.)

**Insight 4 — Thứ tự duyệt.** Cơ số nhỏ hơn ⇔ nhiều chữ số hơn. Chính xác: nếu `k1 < k2` đều là cơ số tốt với `m1, m2` chữ số thì `m1 > m2` (vì `n < (k+1)^(m-1)` và `n >= k^(m-1)` nên `m` xác định bởi `k`, và `m` giảm khi `k` tăng). Vậy duyệt `m` từ **lớn xuống nhỏ**, gặp `m` đầu tiên có cơ số tốt là xong.

Quy tắc: **nếu** với `m` hiện tại, `f(⌊n^(1/(m-1))⌋) == n` **thì** trả về ngay cơ số đó.

In [12]:
# Minh họa Insight 3 với n = 4681, m = 5
n, m = 4681, 5
print("n^(1/(m-1)) ≈", n ** (1 / (m - 1)))
k = iroot(n, m - 1)
print("k = floor =", k, " f(k) =", geo_sum(k, m), " = n?", geo_sum(k, m) == n)

n^(1/(m-1)) ≈ 8.271507355518056
k = floor = 8  f(k) = 4681  = n? True


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt `m` từ lớn xuống + binary search `k`

Với mỗi `m` từ `n.bit_length()` xuống `3`:
- Binary search `k` trong `[2, iroot(n, m-1)]` (cận trên từ Insight 3: `k <= n^(1/(m-1))`).
- So sánh `f(k)` với `n`. Để tránh tính số khổng lồ, khi cộng dồn vượt `n` thì dừng sớm.

Nếu không `m >= 3` nào thành công, trả về `n - 1`.

In [13]:
def compare_geo(k, m, n):
    """So sánh f(k) = 1 + k + ... + k^(m-1) với n: trả -1, 0, 1."""
    total, term = 0, 1
    for _ in range(m):
        total += term
        if total > n:
            return 1
        term *= k
    return 0 if total == n else -1

def good_base_bsearch(n_str):
    n = int(n_str)
    for m in range(n.bit_length(), 2, -1):
        lo, hi = 2, iroot(n, m - 1)
        while lo <= hi:
            mid = (lo + hi) // 2
            c = compare_geo(mid, m, n)
            if c == 0:
                return str(mid)
            if c < 0:
                lo = mid + 1
            else:
                hi = mid - 1
    return str(n - 1)

for s in ["13", "4681", "1000000000000000000", "3"]:
    print(s, "->", good_base_bsearch(s))
assert good_base_bsearch("13") == "3" and good_base_bsearch("4681") == "8"

13 -> 3
4681 -> 8
1000000000000000000 -> 999999999999999999
3 -> 2


**Code nộp LeetCode (Cách 2):**

In [14]:
class SolutionBinarySearch:
    def smallestGoodBase(self, n: str) -> str:
        num = int(n)
        for m in range(num.bit_length(), 2, -1):
            lo, hi = 2, int(num ** (1 / (m - 1))) + 1      # cận trên rộng hơn một chút cho an toàn
            while lo <= hi:
                mid = (lo + hi) // 2
                total, term = 0, 1
                for _ in range(m):
                    total += term
                    if total > num:
                        break
                    term *= mid
                if total == num:
                    return str(mid)
                if total < num:
                    lo = mid + 1
                else:
                    hi = mid - 1
        return str(num - 1)

print(SolutionBinarySearch().smallestGoodBase("4681"))

8


**Độ phức tạp Cách 2:** `m` có `O(log n)` giá trị, binary search `O(log n)` bước, mỗi bước tính tổng `O(m) = O(log n)` → `O(log³ n)` ≈ `60·60·60 = 216 000` phép. Space `O(1)`.

### 7.2. Cách 3 — Chỉ kiểm tra ứng viên duy nhất `k = ⌊n^(1/(m-1))⌋`

Theo Insight 3, không cần binary search: tính thẳng `k = iroot(n, m-1)`, rồi kiểm tra `f(k) == n`. Mỗi `m` tốn `O(log n)` cho căn nguyên + `O(m)` cho tổng.

In [15]:
def good_base_root(n_str):
    n = int(n_str)
    for m in range(n.bit_length(), 2, -1):
        k = iroot(n, m - 1)
        if k >= 2 and geo_sum(k, m) == n:
            return str(k)
    return str(n - 1)

for s in ["13", "4681", "1000000000000000000", "3", "15", "121"]:
    print(s, "->", good_base_root(s))

13 -> 3
4681 -> 8
1000000000000000000 -> 999999999999999999
3 -> 2
15 -> 2
121 -> 3


**Code nộp LeetCode (Cách 3)** — dùng `float` để lấy ứng viên rồi kiểm tra cả `k-1, k, k+1` cho chắc (phòng sai số dấu phẩy động):

In [16]:
class SolutionRoot:
    def smallestGoodBase(self, n: str) -> str:
        num = int(n)
        for m in range(num.bit_length(), 2, -1):
            k0 = int(num ** (1.0 / (m - 1)))
            for k in (k0 - 1, k0, k0 + 1):
                if k < 2:
                    continue
                if (k ** m - 1) // (k - 1) == num:
                    return str(k)
        return str(num - 1)

print(SolutionRoot().smallestGoodBase("1000000000000000000"))

999999999999999999


**Độ phức tạp Cách 3:** `O(log n)` giá trị `m`, mỗi giá trị `O(1)` phép lũy thừa (Python tính `k**m` khoảng `O(log m)` phép nhân số lớn) → thực tế `O(log² n)`. Space `O(1)`.

### 7.3. Cách 4 — Kiểm tra bằng đổi cơ số (biến thể dễ nhớ)

Cùng ứng viên `k` như Cách 3 nhưng kiểm tra bằng `all_ones(n, k)` — đúng định nghĩa đề, không cần công thức cấp số nhân. Thêm điều kiện số chữ số đúng bằng `m` là không cần: nếu toàn chữ số 1 thì `k` đã là cơ số tốt.

In [17]:
def good_base_allones(n_str):
    n = int(n_str)
    for m in range(n.bit_length(), 2, -1):
        k = iroot(n, m - 1)
        if k >= 2 and all_ones(n, k):
            return str(k)
    return str(n - 1)

assert good_base_allones("13") == "3" and good_base_allones("4681") == "8"
print(good_base_allones("1000000000000000000"))

999999999999999999


Code nộp LeetCode cho Cách 4 giống Cách 3, chỉ thay dòng kiểm tra bằng vòng `while num % k == 1`. Độ phức tạp `O(log² n)`.

## 8. TRACE từng bước

Trace **Cách 3** với `n = 4681`. Các cột: `m` (số chữ số đang thử), `k = ⌊n^(1/(m-1))⌋`, `f(k)` và so sánh.

| m | k = iroot(n, m-1) | f(k) | kết quả |
|---|---|---|---|
| 13 | 2 | 8191 | > n |
| 12 | 2 | 4095 | < n |
| … | … | … | … |
| 5 | 8 | 4681 | **= n → trả về 8** |

In [18]:
def trace_root(n_str):
    n = int(n_str)
    print(f"n = {n}, m tối đa = n.bit_length() = {n.bit_length()}")
    for m in range(n.bit_length(), 2, -1):
        k = iroot(n, m - 1)
        f = geo_sum(k, m) if k >= 2 else None
        rel = "k<2" if f is None else ("=" if f == n else ("<" if f < n else ">"))
        print(f"  m={m:>2}  k={k:<6} f(k)={f!s:<8} f(k) {rel} n")
        if f == n:
            print(f"  => trả về {k}")
            return str(k)
    print(f"  => không có m>=3, trả về n-1 = {n-1}")
    return str(n - 1)

trace_root("4681")
print()
trace_root("50")

n = 4681, m tối đa = n.bit_length() = 13
  m=13  k=2      f(k)=8191     f(k) > n
  m=12  k=2      f(k)=4095     f(k) < n
  m=11  k=2      f(k)=2047     f(k) < n
  m=10  k=2      f(k)=1023     f(k) < n
  m= 9  k=2      f(k)=511      f(k) < n
  m= 8  k=3      f(k)=3280     f(k) < n
  m= 7  k=4      f(k)=5461     f(k) > n
  m= 6  k=5      f(k)=3906     f(k) < n
  m= 5  k=8      f(k)=4681     f(k) = n
  => trả về 8

n = 50, m tối đa = n.bit_length() = 6
  m= 6  k=2      f(k)=63       f(k) > n
  m= 5  k=2      f(k)=31       f(k) < n
  m= 4  k=3      f(k)=40       f(k) < n
  m= 3  k=7      f(k)=57       f(k) > n
  => không có m>=3, trả về n-1 = 49


'49'

Trace **Cách 2** (binary search) với `n = 13`, `m = 3` để thấy khoảng `[lo, hi]` co lại:

In [19]:
def trace_bsearch(n, m):
    lo, hi = 2, iroot(n, m - 1)
    step = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        c = compare_geo(mid, m, n)
        step += 1
        print(f"bước {step}: lo={lo} hi={hi} mid={mid} f(mid) {'<' if c<0 else ('=' if c==0 else '>')} n")
        if c == 0:
            return mid
        lo, hi = (mid + 1, hi) if c < 0 else (lo, mid - 1)
    return None

print("kết quả:", trace_bsearch(13, 3))
print("kết quả:", trace_bsearch(10**18 - 1, 3))

bước 1: lo=2 hi=3 mid=2 f(mid) < n
bước 2: lo=3 hi=3 mid=3 f(mid) = n
kết quả: 3
bước 1: lo=2 hi=999999999 mid=500000000 f(mid) < n
bước 2: lo=500000001 hi=999999999 mid=750000000 f(mid) < n
bước 3: lo=750000001 hi=999999999 mid=875000000 f(mid) < n
bước 4: lo=875000001 hi=999999999 mid=937500000 f(mid) < n
bước 5: lo=937500001 hi=999999999 mid=968750000 f(mid) < n
bước 6: lo=968750001 hi=999999999 mid=984375000 f(mid) < n
bước 7: lo=984375001 hi=999999999 mid=992187500 f(mid) < n
bước 8: lo=992187501 hi=999999999 mid=996093750 f(mid) < n
bước 9: lo=996093751 hi=999999999 mid=998046875 f(mid) < n
bước 10: lo=998046876 hi=999999999 mid=999023437 f(mid) < n
bước 11: lo=999023438 hi=999999999 mid=999511718 f(mid) < n
bước 12: lo=999511719 hi=999999999 mid=999755859 f(mid) < n
bước 13: lo=999755860 hi=999999999 mid=999877929 f(mid) < n
bước 14: lo=999877930 hi=999999999 mid=999938964 f(mid) < n
bước 15: lo=999938965 hi=999999999 mid=999969482 f(mid) < n
bước 16: lo=999969483 hi=999999999 m

## 9. Code nộp LeetCode

Bản nộp chính: Cách 3 với căn nguyên chính xác (không phụ thuộc `float`).

In [20]:
class Solution:
    def smallestGoodBase(self, n: str) -> str:
        num = int(n)

        def iroot(x, e):
            lo, hi = 1, 1 << (x.bit_length() // e + 1)
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if mid ** e <= x:
                    lo = mid
                else:
                    hi = mid - 1
            return lo

        for m in range(num.bit_length(), 2, -1):    # m = số chữ số 1
            k = iroot(num, m - 1)
            if k >= 2 and (k ** m - 1) // (k - 1) == num:
                return str(k)
        return str(num - 1)                          # luôn đúng: n = "11" trong cơ số n-1

sol = Solution()
assert sol.smallestGoodBase("13") == "3"
assert sol.smallestGoodBase("4681") == "8"
assert sol.smallestGoodBase("1000000000000000000") == "999999999999999999"
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

1. **Mọi cơ số tốt `k` ứng với đúng một `m` (số chữ số).** Ta có `k^(m-1) <= n < k^m` (số có `m` chữ số trong cơ số `k`).
2. **`m` nằm trong `[2, n.bit_length()]`:** vì `n >= 2^(m-1)` (do `k >= 2`) → `m - 1 <= log2 n`.
3. **Với mỗi `m >= 3`, nếu có cơ số tốt thì nó bằng `⌊n^(1/(m-1))⌋`** (Insight 3, chứng minh bằng khai triển nhị thức: `k^(m-1) < n < (k+1)^(m-1)`). Vậy kiểm tra một ứng viên là đủ, không bỏ sót.
4. **Cơ số nhỏ hơn ⇔ `m` lớn hơn.** Giả sử `k1 < k2` là hai cơ số tốt, với `m1`, `m2` chữ số. Từ (3): `k2^(m2-1) < n < (k2+1)^(m2-1) <= ... ` Cụ thể nếu `m1 <= m2` thì `n = f_{m1}(k1) <= f_{m2}(k1) < f_{m2}(k2) = n` — mâu thuẫn (hàm `f` tăng theo cả `k` và `m`). Do đó `m1 > m2`. Vì thế duyệt `m` giảm dần, cơ số đầu tiên tìm được là nhỏ nhất.
5. **`m = 2` luôn có nghiệm `k = n - 1`** (vì `n >= 3` nên `n - 1 >= 2`), nên vòng lặp luôn có đáp án.

### 10.1. Kiểm chứng bằng máy các bổ đề

Chứng minh trên giấy là chính, nhưng kiểm tra bằng máy giúp ta tin tưởng hơn và phát hiện lỗi lập luận:

- Bổ đề A: `k^(m-1) < f(k, m) < (k+1)^(m-1)` với mọi `k >= 2`, `m >= 3`.
- Bổ đề B: nếu `f(k1, m1) == f(k2, m2)` với `k1 < k2` thì `m1 > m2`.

In [21]:
for k in range(2, 300):
    for m in range(3, 25):
        f = geo_sum(k, m)
        assert k ** (m - 1) < f < (k + 1) ** (m - 1), (k, m)
print("Bổ đề A đúng với k < 300, 3 <= m < 25")

seen = {}
for k in range(2, 2000):
    for m in range(2, 40):
        f = geo_sum(k, m)
        if f > 10**12:
            break
        seen.setdefault(f, []).append((k, m))
for f, lst in seen.items():
    lst.sort()
    for (k1, m1), (k2, m2) in zip(lst, lst[1:]):
        assert m1 > m2, (f, lst)
print("Bổ đề B đúng với mọi n <= 10^12 sinh từ k < 2000; ví dụ:", seen[31], seen[8191])

Bổ đề A đúng với k < 300, 3 <= m < 25
Bổ đề B đúng với mọi n <= 10^12 sinh từ k < 2000; ví dụ: [(2, 5), (5, 3), (30, 2)] [(2, 13), (90, 3)]


## 11. Phân tích Time/Space

- Số giá trị `m`: tối đa 60.
- Cách 2: mỗi `m` binary search ~60 bước × tính tổng ≤60 phép → ≤ `60³ ≈ 2·10^5`. **Time `O(log³ n)`**, Space `O(1)`.
- Cách 3: mỗi `m` một lần căn nguyên (`O(log n)` bước × lũy thừa) + một lần kiểm tra. Thực tế **`O(log² n)`** phép số học trên số ≤ `10^18`-ish (Python số lớn, coi như hằng). Space `O(1)`.
- Brute force: `O(n log n)` — vô vọng.

In [22]:
import time
for name, f in [("bsearch", good_base_bsearch), ("root", good_base_root), ("Solution", Solution().smallestGoodBase)]:
    t = time.perf_counter()
    for n in range(10**18 - 200, 10**18 + 1):
        f(str(n))
    print(f"{name:<9}: 201 lần gọi với n ≈ 10^18 mất {time.perf_counter() - t:.3f}s")

bsearch  : 201 lần gọi với n ≈ 10^18 mất 0.043s


root     : 201 lần gọi với n ≈ 10^18 mất 0.020s


Solution : 201 lần gọi với n ≈ 10^18 mất 0.008s


## 12. Bộ test

In [23]:
methods = {
    "brute": good_base_brute,
    "bsearch": good_base_bsearch,
    "root": good_base_root,
    "allones": good_base_allones,
    "SolBS": SolutionBinarySearch().smallestGoodBase,
    "SolRoot": SolutionRoot().smallestGoodBase,
    "Solution": Solution().smallestGoodBase,
}
cases = {
    "13": "3", "4681": "8", "1000000000000000000": "999999999999999999",
    "3": "2",            # nhỏ nhất: 3 = 11 cơ số 2
    "7": "2",            # 111 cơ số 2
    "15": "2",           # 1111 cơ số 2
    "21": "4",           # 111 cơ số 4
    "4": "3",            # chỉ có 11 cơ số 3
}
for s, exp in cases.items():
    for name, f in methods.items():
        if name == "brute" and int(s) > 10**6:
            continue
        assert f(s) == exp, (name, s, f(s), exp)
print("Ví dụ đề + edge case: OK")

Ví dụ đề + edge case: OK


In [24]:
# Edge case lớn: n = 2^60 - 1 (toàn 1 trong cơ số 2, 60 chữ số); và số dạng (k^m-1)/(k-1) lớn
big_cases = {str(2**60 - 1): "2", str((3**37 - 1) // 2): "3", str(10**9 * 10**9 + 10**9 + 1): str(10**9)}
for s, exp in big_cases.items():
    for name, f in methods.items():
        if name == "brute":
            continue
        assert f(s) == exp, (name, s, f(s), exp)
print("Edge case lớn: OK")

Edge case lớn: OK


In [25]:
# Đối chiếu ngẫu nhiên với brute force (n nhỏ) và giữa các cách (n lớn)
import random
random.seed(483)
for n in range(3, 3000):
    exp = good_base_brute(str(n))
    for name, f in methods.items():
        assert f(str(n)) == exp, (name, n)
for _ in range(2000):
    # tạo n có cơ số tốt "đẹp" với xác suất cao
    k = random.randint(2, 10**4)
    m = random.randint(2, 6)
    n = (k ** m - 1) // (k - 1)
    if not (3 <= n <= 10**18):
        continue
    answers = {name: f(str(n)) for name, f in methods.items() if name != "brute"}
    assert len(set(answers.values())) == 1, (n, answers)
    assert int(answers["Solution"]) <= k
for _ in range(2000):
    n = random.randint(3, 10**18)
    answers = {name: f(str(n)) for name, f in methods.items() if name != "brute"}
    assert len(set(answers.values())) == 1, (n, answers)
    assert all_ones(n, int(answers["Solution"]))
print("Test ngẫu nhiên (seed 483): OK")

Test ngẫu nhiên (seed 483): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tin hoàn toàn vào `float` khi lấy căn

**Code sai:** `k = int(n ** (1/(m-1)))` rồi chỉ kiểm tra đúng `k` đó.

**Vì sao sai:** `n ** 0.5` với `n` ~ `10^36` hoặc `10^18` có thể lệch 1 đơn vị (ví dụ ra `k - 1` khi giá trị thật là `k` nhưng bị làm tròn xuống `k - 0.0000001`).

In [26]:
n = 999999999999999999 ** 2 + 999999999999999999 + 1   # = f(k=999999999999999999, m=3) > 10^18 nhưng minh họa lỗi float
k_float = int(n ** 0.5)
print("float cho:", k_float, "| iroot cho:", iroot(n, 2))
# Code đúng: dùng iroot, hoặc thử cả k-1, k, k+1 như SolutionRoot

float cho: 1000000000000000000 | iroot cho: 999999999999999999


### Lỗi 2 — Duyệt `m` từ nhỏ lên lớn

**Code sai:** `for m in range(2, 61)` rồi trả về ngay khi gặp nghiệm → luôn trả về `n-1` ở `m = 2`!

**Vì sao sai:** `m` nhỏ ⇔ cơ số lớn. Phải duyệt `m` **giảm dần**.

In [27]:
def wrong_order(n_str):
    n = int(n_str)
    for m in range(2, n.bit_length() + 1):
        k = iroot(n, m - 1)
        if k >= 2 and geo_sum(k, m) == n:
            return str(k)
    return str(n - 1)

print("sai:", wrong_order("13"), " đúng:", Solution().smallestGoodBase("13"))

sai: 3  đúng: 3


### Lỗi 3 — Tính `f(k)` không dừng sớm trong binary search

**Code sai:** `sum(mid**i for i in range(m))` với `mid` lớn và `m = 60` → tạo số cỡ `10^1000`, chậm hẳn (vẫn đúng trong Python nhưng tràn số trong C++/Java).

**Code đúng:** cộng dồn và `break` ngay khi `total > n` (hàm `compare_geo`).

### Lỗi 4 — Quên trường hợp `m = 2` / trả về số thay vì chuỗi

- Nếu vòng lặp chạy `m` xuống tới 2 với `iroot(n, 1) = n` thì `geo_sum(n, 2) = n + 1 ≠ n` — sẽ **không** tìm thấy. Phải trả về `n - 1` riêng.
- LeetCode yêu cầu kiểu `str`: `return k` (int) sẽ bị chấm sai.

In [28]:
n = 13
print("iroot(n,1) =", iroot(n, 1), "-> geo_sum =", geo_sum(iroot(n, 1), 2), "≠", n, "; đáp án m=2 phải là n-1 =", n - 1)
print(type(Solution().smallestGoodBase("13")))

iroot(n,1) = 13 -> geo_sum = 14 ≠ 13 ; đáp án m=2 phải là n-1 = 12
<class 'str'>


## 14. Mẹo & nhận dạng pattern

- Đề có **"cơ số"** + **"tất cả chữ số giống nhau / là 1"** → nghĩ tới **tổng cấp số nhân**.
- Không gian một biến quá lớn (`k` tới `10^18`) nhưng biến kia rất nhỏ (`m <= 60`) → **đổi biến duyệt**.
- Hàm đơn điệu theo biến cần tìm → **binary search trên đáp án**.
- Có công thức kẹp `k^(m-1) < n < (k+1)^(m-1)` → ứng viên duy nhất `⌊n^(1/(m-1))⌋`.
- Làm việc với số tới `10^18`: cẩn thận `float` (53 bit mantissa ≈ `9·10^15`). Dùng căn nguyên `iroot` hoặc `math.isqrt` (cho căn bậc 2).

### 14.1. Khuôn binary search trên đáp án (dùng lại được)

Khi có hàm **đơn điệu tăng** `g(x)` và cần tìm `x` sao cho `g(x) == target`:

```python
lo, hi = <cận dưới>, <cận trên>
while lo <= hi:
    mid = (lo + hi) // 2
    if g(mid) == target: return mid
    if g(mid) < target: lo = mid + 1
    else:               hi = mid - 1
return None   # không tồn tại
```

Ở bài này `g(k) = f(k, m)` với `m` cố định. Cận trên `iroot(n, m-1)` đến từ `k^(m-1) < n`.

In [29]:
def bsearch_eq(g, target, lo, hi):
    while lo <= hi:
        mid = (lo + hi) // 2
        v = g(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return None

print(bsearch_eq(lambda k: geo_sum(k, 5), 4681, 2, 4681))   # 8
print(bsearch_eq(lambda x: x * x, 144, 0, 144))              # 12
print(bsearch_eq(lambda x: x * x, 145, 0, 145))              # None

8
12
None


## 15. Biến thể

1. **Cơ số tốt lớn nhất khác `n-1`:** duyệt `m` tăng dần từ 3.
2. **Tất cả cơ số tốt của `n`:** duyệt mọi `m`, thu tất cả `k`.
3. **"Repdigit" tổng quát:** `n` viết toàn chữ số `d` trong cơ số `k` → `n = d · f(k)`, thử mọi ước `d < k`.
4. **Đếm số `n <= N` có cơ số tốt khác `n-1`** (Project-Euler-style): sinh mọi `f(k, m)` với `m >= 3`.

Demo biến thể 2:

In [30]:
def all_good_bases(n):
    res = []
    for m in range(n.bit_length(), 2, -1):
        k = iroot(n, m - 1)
        if k >= 2 and geo_sum(k, m) == n:
            res.append(k)
    res.append(n - 1)
    return res

print(all_good_bases(31))       # 31 = 11111 (2) = 111 (5) = 11 (30)
print(all_good_bases(8191))     # 8191 = 2^13-1 = 111 (90)
assert all_good_bases(31) == [2, 5, 30]

[2, 5, 30]
[2, 90, 8190]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Thử mọi `k`, đổi cơ số | `O(n log n)` | `O(1)` | Kiểm chứng `n` nhỏ |
| 2. Binary search | Duyệt `m` giảm, tìm `k` nhị phân | `O(log³ n)` | `O(1)` | Không nhớ công thức kẹp, vẫn nhanh |
| 3. Ứng viên duy nhất | `k = ⌊n^(1/(m-1))⌋`, kiểm tra | `O(log² n)` | `O(1)` | Lời giải gọn nhất |
| 4. Ứng viên + đổi cơ số | Như 3 nhưng kiểm tra bằng `%` | `O(log² n)` | `O(1)` | Dễ đọc, sát định nghĩa |

## 17. Bài tập

**Bài 1 (đoán output).** `to_base(31, 5)` trả về gì?

**Bài 2 (tính tay).** Tìm cơ số tốt nhỏ nhất của `n = 57`. (Gợi ý: thử `m = 3`: `k² + k + 1 = 57`.)

**Bài 3 (điền TODO).** Hoàn thiện `compare_geo_todo`.

In [31]:
def compare_geo_todo(k, m, n):
    total, term = 0, 1
    for _ in range(m):
        # TODO: cộng term vào total; nếu total > n thì trả 1; nhân term với k
        pass
    # TODO: trả 0 nếu total == n, ngược lại -1
    return None

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert compare_geo_todo(3, 3, 13) == 0

**Bài 4 (tự viết lại).** Viết lại `Solution` theo Cách 2 (binary search) mà không nhìn code.

**Bài 5 (mini challenge).** Viết `smallest_repdigit_base(n)`: cơ số `k >= 2` nhỏ nhất sao cho `n` viết trong cơ số `k` có **mọi chữ số bằng nhau** và có **ít nhất 2 chữ số** (chữ số có thể khác 1). Gợi ý: `n = d · f(k, m)` với `1 <= d < k`.

In [32]:
def smallest_repdigit_base_todo(n):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Dùng `%` và `//` lặp lại.
- Mức 2: `57 = 49 + 7 + 1`.
- Mức 3: Bài 2: `k = 7`.

**Bài 3**
- Mức 1: Ba dòng trong vòng lặp.
- Mức 2: Kiểm tra vượt **sau** khi cộng, **trước** khi nhân.
- Mức 3: `return 0 if total == n else -1`.

**Bài 4**
- Mức 1: Vòng ngoài `m` giảm dần từ `bit_length`.
- Mức 2: `hi = iroot(n, m-1)`.
- Mức 3: Cuối cùng trả `str(n-1)`.

**Bài 5**
- Mức 1: Brute force trước để có đáp án đối chiếu.
- Mức 2: Ứng với mỗi `k`, `n` toàn chữ số `d` ⇔ `n % k == d` ở mọi bước và `d = n % k`.
- Mức 3: Với `k <= √n` kiểm tra trực tiếp; với `k > √n` số chỉ có 2 chữ số `dd`, tức `n = d·(k+1)` với `d < k` — duyệt ước `d <= √n`.

## 19. Đáp án tham khảo

In [33]:
# Bài 1
assert to_base(31, 5) == [1, 1, 1]
# Bài 2
assert Solution().smallestGoodBase("57") == "7" and to_base(57, 7) == [1, 1, 1]

# Bài 3
def compare_geo_sol(k, m, n):
    total, term = 0, 1
    for _ in range(m):
        total += term
        if total > n:
            return 1
        term *= k
    return 0 if total == n else -1
assert compare_geo_sol(3, 3, 13) == 0 and compare_geo_sol(3, 4, 13) == 1 and compare_geo_sol(2, 3, 13) == -1

# Bài 4
class SolutionEx4:
    def smallestGoodBase(self, n: str) -> str:
        num = int(n)
        for m in range(num.bit_length(), 2, -1):
            lo, hi = 2, iroot(num, m - 1)
            while lo <= hi:
                mid = (lo + hi) // 2
                c = compare_geo_sol(mid, m, num)
                if c == 0:
                    return str(mid)
                lo, hi = (mid + 1, hi) if c < 0 else (lo, mid - 1)
        return str(num - 1)
for n in range(3, 2000):
    assert SolutionEx4().smallestGoodBase(str(n)) == good_base_brute(str(n))
print("Bài 1–4 OK")

Bài 1–4 OK


In [34]:
# Bài 5
def repdigit(n, k):
    d = n % k
    if d == 0 or n < k:          # cần >= 2 chữ số, chữ số đầu khác 0
        return False
    while n > 0:
        if n % k != d:
            return False
        n //= k
    return True

def smallest_repdigit_base_brute(n):
    for k in range(2, n):
        if repdigit(n, k):
            return k

def smallest_repdigit_base(n):
    import math
    r = math.isqrt(n)
    # Với k <= r+1: kiểm tra trực tiếp (O(sqrt n) cơ số, mỗi cơ số O(log n))
    for k in range(2, r + 2):
        if repdigit(n, k):
            return k
    # Với k > r+1 thì k^2 > n nên n chỉ có 2 chữ số: n = d*k + d = d*(k+1), 1 <= d < k.
    # d < k và d*(k+1) = n  =>  d*d < n  =>  d <= r. Chọn d lớn nhất -> k nhỏ nhất.
    best = n - 1
    for d in range(1, r + 1):
        if n % d == 0 and d < n // d - 1:
            best = min(best, n // d - 1)
    return best

for n in range(3, 1500):
    assert smallest_repdigit_base(n) == smallest_repdigit_base_brute(n), n
print("Bài 5 OK, ví dụ:", smallest_repdigit_base(26), to_base(26, smallest_repdigit_base(26)))

Bài 5 OK, ví dụ: 3 [2, 2, 2]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
num = int(n)
for m in range(num.bit_length(), 2, -1):        # nhiều chữ số -> cơ số nhỏ
    k = iroot(num, m - 1)                        # ứng viên duy nhất
    if k >= 2 and (k**m - 1) // (k - 1) == num:
        return str(k)
return str(num - 1)                              # luôn "11"
```

**Checklist trước khi submit:**

- [ ] Duyệt `m` **giảm dần** từ `num.bit_length()`.
- [ ] Không tin `float` mù quáng: dùng căn nguyên hoặc thử `k-1, k, k+1`.
- [ ] Trường hợp `m = 2` trả về `n - 1` riêng.
- [ ] Trả về **chuỗi** (`str`).
- [ ] Đã thử `n = 3`, `n = 2^60 - 1`, `n = 10^18`, các số dạng `(k^m-1)/(k-1)`.
- [ ] Nếu dùng binary search: dừng sớm khi tổng vượt `n`.